# AI Capstone: ผู้ช่วยตอบคำถามจากฐานความรู้ (RAG)

Notebook เริ่มต้นจาก Data Career Lab — อ่าน rubric ในบทเรียน `AI Capstone` บนเว็บก่อน แล้วเติมเซลล์ที่มี `TODO`

- เซลล์ทั้งหมดรันได้ **ออฟไลน์** (ไม่ต้องมี API key) ด้วยตัวสร้างคำตอบสำรองแบบสกัดประโยค
- ถ้าตั้ง `ANTHROPIC_API_KEY` (ตัวแปรสภาพแวดล้อมหรือ Colab Secrets) จะเรียกโมเดลจริงในหัวข้อ 4
- **ห้ามพิมพ์ API key ลงใน notebook หรือ commit ขึ้น GitHub**

In [ ]:
import os
if not os.path.exists('data-career-lab'):
    os.system('git clone https://github.com/Phakinza007/data-career-lab.git')
os.chdir('data-career-lab')

import math, re, json
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS
from sklearn.metrics.pairwise import cosine_similarity

docs = pd.read_csv('public/data/kb_docs.csv')
qs = pd.read_csv('public/data/kb_questions.csv')
print(len(docs), 'เอกสาร,', len(qs), 'คำถามที่มีเฉลย')
docs.head(3)

## 1. Retrieval baseline (TF-IDF)

In [ ]:
vec = TfidfVectorizer(stop_words='english')
D = vec.fit_transform(docs['text'])


def retrieve(question, k=3):
    s = cosine_similarity(vec.transform([question]), D)[0]
    order = np.argsort(-s, kind='stable')[:k]
    return [(docs['doc_id'][j], float(s[j])) for j in order]


sims = cosine_similarity(vec.transform(qs['question']), D)
ranks = np.array([list(docs['doc_id'].values[np.argsort(-sims[i], kind='stable')]).index(g) + 1 for i, g in enumerate(qs['gold_doc_id'])])
print('hit@1', round((ranks == 1).mean(), 4), 'hit@3', round((ranks <= 3).mean(), 4), 'MRR', round((1 / ranks).mean(), 4))

## 2. ปรับปรุง retrieval (TODO)

ลองอย่างน้อย 1 อย่าง แล้ววัดด้วยตัวเลขเดิม: chunk/ใส่ title, bigram, query expansion, embedding model จริง, hybrid (ผสมคะแนน) — ดูรายข้อที่เปลี่ยน ไม่ใช่แค่ตัวเลขรวม และเขียนชุดคำถามใหม่ไว้ตรวจสุดท้าย

In [ ]:
# TODO: ลองวิธีปรับปรุง retrieval แล้วเทียบ hit@1 / hit@3 / MRR กับ baseline

## 3. ประกอบ prompt (ยึดบริบท + อ้างอิง + ปฏิเสธ)

In [ ]:
TEMPLATE = (
    "Answer the question using ONLY the context below. Cite the doc ids in square brackets. "
    "If the answer is not in the context, reply exactly: I don't know.\n\n"
    "Context:\n{context}\n\nQuestion: {question}\nAnswer:"
)
rows = docs.set_index('doc_id')


def build_prompt(question, k=3):
    hits = retrieve(question, k)
    context = '\n'.join(f"[{d}] {rows.loc[d, 'title']}: {rows.loc[d, 'text']}" for d, _ in hits)
    return TEMPLATE.format(context=context, question=question), [d for d, _ in hits]


prompt, retrieved = build_prompt(qs['question'][0])
print(prompt)
print('ประมาณ token:', math.ceil(len(prompt) / 4))

## 4. สร้างคำตอบ

ถ้ามี `ANTHROPIC_API_KEY` จะเรียก API จริง (ต้อง `pip install anthropic`) มิฉะนั้นใช้ตัวสำรองแบบสกัดประโยค (ไม่ฉลาดเท่า LLM แต่ทำให้ pipeline ทั้งเส้นรันและวัดผลได้)

In [ ]:
def get_api_key():
    key = os.environ.get('ANTHROPIC_API_KEY')
    if key:
        return key
    try:  # Colab Secrets
        from google.colab import userdata
        return userdata.get('ANTHROPIC_API_KEY')
    except Exception:
        return None


API_KEY = get_api_key()


def sentences(text):
    return [s.strip() for s in re.split(r'(?<=[.;])\s+', text) if s.strip()]


def content_words(text):
    return {w for w in re.findall(r'[a-z0-9]+', text.lower()) if w not in ENGLISH_STOP_WORDS}


def offline_generate(question, doc_ids):
    best, best_score = None, -1
    for d in doc_ids:
        for s in sentences(rows.loc[d, 'text']):
            score = len(content_words(question) & content_words(s))
            if score > best_score:
                best, best_score = (d, s), score
    return f'{best[1]} [{best[0]}]'


if API_KEY:
    import anthropic
    client = anthropic.Anthropic(api_key=API_KEY)

    def generate(prompt, question, doc_ids):
        reply = client.messages.create(model='claude-sonnet-5', max_tokens=300, messages=[{'role': 'user', 'content': prompt}])
        return reply.content[0].text
else:
    def generate(prompt, question, doc_ids):
        return offline_generate(question, doc_ids)

print('โหมด:', 'API จริง' if API_KEY else 'ออฟไลน์ (สกัดประโยค)')

## 5. ตรวจผลลัพธ์และ fallback

id ที่อ้างต้องอยู่ในชุดที่ดึงมา ถ้าไม่ผ่านให้ใช้ fallback (เช่น ส่งต่อให้คน)

In [ ]:
FALLBACK = "I'm not sure. Let me pass this to a human agent."


def answer(question, k=3):
    prompt, retrieved = build_prompt(question, k)
    text = generate(prompt, question, retrieved)
    cited = re.findall(r'\[(d\d+)\]', text)
    if text.strip() != "I don't know." and (not cited or any(c not in retrieved for c in cited)):
        return FALLBACK, retrieved, False
    return text, retrieved, True


for q in qs['question'][:3]:
    print(q, '->', answer(q)[0])

## 6. ประเมินคำตอบ (TODO)

วัดอย่างน้อย: (1) สัดส่วนที่ตอบโดยอ้างเอกสารเฉลย (2) สัดส่วนคำถามนอกขอบเขตที่ปฏิเสธถูก (เขียนอย่างน้อย 6 ข้อเอง) (3) ต้นทุนโดยประมาณต่อคำถาม (ระบุว่าใช้ราคาสมมติหรือราคาจริง) แล้วดูรายข้อที่พลาด

In [ ]:
# TODO: วัดอัตราการอ้างเอกสารเฉลย, การปฏิเสธคำถามนอกขอบเขต และต้นทุน
hits = 0
for q, gold in zip(qs['question'], qs['gold_doc_id']):
    text, retrieved, ok = answer(q)
    hits += ok and gold in re.findall(r'\[(d\d+)\]', text)
print('อ้างเอกสารเฉลย:', round(hits / len(qs), 4))

## 7. สรุป

เขียน 5–8 บรรทัด: ระบบทำอะไร ตัวเลขประเมิน (เทียบ baseline) จุดที่พลาดบ่อยและสาเหตุ ต้นทุน ข้อจำกัด (ชุดคำถามเล็ก ภาษาอังกฤษเท่านั้น ฯลฯ) และสิ่งที่จะทำต่อ

In [ ]:
# เขียนสรุปเป็นคอมเมนต์ตรงนี้